In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
(full_train_images, full_train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
full_train_labels = full_train_labels.flatten()
test_labels = test_labels.flatten()
class_names = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

channel_mean = full_train_images[:45000].mean(axis=(0, 1, 2))
channel_std = full_train_images[:45000].std(axis=(0, 1, 2))

def prepare(images):
    return ((images.astype("float32") - channel_mean) / channel_std).reshape(len(images), -1)

train_features, val_features = prepare(full_train_images[:45000]), prepare(full_train_images[45000:])
train_labels, val_labels = full_train_labels[:45000], full_train_labels[45000:]
test_features = prepare(test_images)
print(train_features.shape, val_features.shape, test_features.shape)
print("feature mean after normalization:", train_features.mean().round(4), "std:", train_features.std().round(4))

In [ ]:
hidden_sizes = [1024, 512, 256]

def build_mlp(use_dropout=False, use_batchnorm=False, dropout_rate=0.3, learning_rate=1e-3):
    model = keras.Sequential([layers.Input(shape=(3072,))])
    for units in hidden_sizes:
        model.add(layers.Dense(units, use_bias=not use_batchnorm))
        if use_batchnorm:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation("relu"))
        if use_dropout:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

build_mlp(use_dropout=True, use_batchnorm=True).summary()

In [ ]:
configurations = {
    "plain MLP": {"use_dropout": False, "use_batchnorm": False},
    "dropout": {"use_dropout": True, "use_batchnorm": False},
    "batch norm": {"use_dropout": False, "use_batchnorm": True},
    "dropout + batch norm": {"use_dropout": True, "use_batchnorm": True}
}

experiment_histories = {}
trained_models = {}
for name, settings in configurations.items():
    keras.utils.set_random_seed(42)
    model = build_mlp(**settings)
    history = model.fit(train_features, train_labels, validation_data=(val_features, val_labels), epochs=30, batch_size=256, verbose=0)
    experiment_histories[name] = history.history
    trained_models[name] = model
    print(f"{name:<22} train acc {history.history['accuracy'][-1]:.4f}  val acc {history.history['val_accuracy'][-1]:.4f}  best val acc {max(history.history['val_accuracy']):.4f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 9), sharey="row")
for column, (name, history) in enumerate(experiment_histories.items()):
    axes[0, column].plot(history["accuracy"], label="train")
    axes[0, column].plot(history["val_accuracy"], label="validation")
    axes[0, column].set_title(f"{name} accuracy")
    axes[0, column].legend()
    axes[1, column].plot(history["loss"], label="train")
    axes[1, column].plot(history["val_loss"], label="validation")
    axes[1, column].set_title(f"{name} loss")
    axes[1, column].set_xlabel("epoch")
    axes[1, column].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for name, history in experiment_histories.items():
    axes[0].plot(history["val_accuracy"], label=name)
    axes[1].plot(history["val_loss"], label=name)
axes[0].set_title("Validation accuracy")
axes[1].set_title("Validation loss")
for axis in axes:
    axis.set_xlabel("epoch")
    axis.legend()
plt.show()

In [ ]:
result_rows = []
for name, history in experiment_histories.items():
    test_loss, test_accuracy = trained_models[name].evaluate(test_features, test_labels, verbose=0)
    result_rows.append({
        "model": name,
        "final train accuracy": history["accuracy"][-1],
        "final val accuracy": history["val_accuracy"][-1],
        "generalization gap": history["accuracy"][-1] - history["val_accuracy"][-1],
        "lowest val loss": min(history["val_loss"]),
        "test accuracy": test_accuracy
    })
results_frame = pd.DataFrame(result_rows).set_index("model").round(4)
display(results_frame)

results_frame[["final train accuracy", "final val accuracy", "test accuracy"]].plot(kind="bar", figsize=(10, 4.5), rot=0)
plt.title("Effect of regularization")
plt.ylim(0, 1)
plt.show()

In [ ]:
dropout_rate_histories = {}
for dropout_rate in [0.1, 0.2, 0.3, 0.5]:
    keras.utils.set_random_seed(42)
    model = build_mlp(use_dropout=True, use_batchnorm=True, dropout_rate=dropout_rate)
    history = model.fit(train_features, train_labels, validation_data=(val_features, val_labels), epochs=20, batch_size=256, verbose=0)
    dropout_rate_histories[dropout_rate] = history.history
    print(f"dropout {dropout_rate}: best val acc {max(history.history['val_accuracy']):.4f}  gap {history.history['accuracy'][-1] - history.history['val_accuracy'][-1]:.4f}")

plt.figure(figsize=(9, 4.5))
for dropout_rate, history in dropout_rate_histories.items():
    plt.plot(history["val_accuracy"], label=f"dropout {dropout_rate}")
plt.title("Validation accuracy for different dropout rates (with batch norm)")
plt.xlabel("epoch")
plt.legend()
plt.show()

In [ ]:
keras.utils.set_random_seed(42)
probe_model = trained_models["batch norm"]
probe_inputs = val_features[:1000]
dense_layers = [layer for layer in probe_model.layers if isinstance(layer, layers.Dense)][:3]
bn_layers = [layer for layer in probe_model.layers if isinstance(layer, layers.BatchNormalization)]
probe = keras.Model(probe_model.inputs, [layer.output for layer in dense_layers] + [layer.output for layer in bn_layers])
probe_outputs = probe.predict(probe_inputs, verbose=0)

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for column in range(3):
    axes[0, column].hist(probe_outputs[column].flatten(), bins=80, color="gray")
    axes[0, column].set_title(f"dense layer {column + 1} before batch norm")
    axes[1, column].hist(probe_outputs[column + 3].flatten(), bins=80, color="seagreen")
    axes[1, column].set_title(f"dense layer {column + 1} after batch norm")
plt.tight_layout()
plt.show()

In [ ]:
keras.utils.set_random_seed(42)
final_model = build_mlp(use_dropout=True, use_batchnorm=True, dropout_rate=0.3)
final_history = final_model.fit(
    train_features, train_labels,
    validation_data=(val_features, val_labels),
    epochs=80,
    batch_size=256,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5)
    ]
)
final_loss, final_accuracy = final_model.evaluate(test_features, test_labels, verbose=0)
print(f"final regularized MLP test accuracy: {final_accuracy:.4f}")

In [ ]:
final_predictions = final_model.predict(test_features, verbose=0).argmax(axis=1)
print(classification_report(test_labels, final_predictions, target_names=class_names, digits=4))

plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(test_labels, final_predictions), annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("predicted")
plt.ylabel("actual")
plt.show()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for axis, index in zip(axes.flatten(), np.random.choice(len(test_images), 16, replace=False)):
    axis.imshow(test_images[index])
    color = "green" if final_predictions[index] == test_labels[index] else "red"
    axis.set_title(f"{class_names[final_predictions[index]]}\n({class_names[test_labels[index]]})", color=color, fontsize=9)
    axis.axis("off")
plt.tight_layout()
plt.show()

final_model.save("regularized_mlp.keras")